# Stage 3: what we predict, and the Naive baseline

Three jobs in this notebook:
1. Turn "how wiggly was the week" into **one number** per stock per day (the *wiggle*).
2. Build the lazy opponent, the **Naive baseline**: "next week will be as wiggly as last week".
3. Score Naive on an **exam period** it has no way of knowing about.

Run the cells top to bottom, one at a time. The kernel at the top right should say **RiskWise (.venv)**.

In [ ]:
# Cell 1: load the five big stocks we downloaded in Stage 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("../data/prices")
STOCKS = ["RELIANCE", "TCS", "INFY", "HDFCBANK", "SBIN"]

prices = {}
for s in STOCKS:
    df = pd.read_csv(DATA / f"{s}.csv", index_col="date", parse_dates=True)
    no_trades = df["Volume"] == 0       # Yahoo sometimes adds a row for a market holiday: nothing traded, price just copied
    print(f"{s:9s} {len(df)} rows, dropping {int(no_trades.sum())} zero-volume rows")
    prices[s] = df[~no_trades]
prices["RELIANCE"].tail(3)

**Why we dropped zero-volume rows:** a day with no trades at all is not a real trading day. For these big stocks the only cause is a market holiday that the data source filled in. For thinly traded stocks it is different: a zero-volume day can be real, because nobody traded that day. We will need a separate rule for those later.

## The simple wiggle (our cross-check)

A **daily move** is how much the price changed since yesterday, in percent. (We use the logarithm version, which adds up more cleanly. For small moves it is almost the same as the plain percent.)

The simple wiggle ignores direction: square each daily move (this removes minus signs), average them over the last 5 days, then take the square root to undo the squaring. For moves of +1, -2, +0.5, -1, +1.5 that gives about 1.3% a day.

In [ ]:
# Cell 2: the simple wiggle
def simple_wiggle(df, n=5):
    r = np.log(df["Close"]).diff() * 100           # daily move in percent
    return np.sqrt((r ** 2).rolling(n).mean())     # square, average over n days, square root

simple_wiggle(prices["RELIANCE"]).tail(3).round(2)

## The Yang-Zhang wiggle (our main measure)

The simple wiggle looks at each day's **closing price only**. A day when the price swung 4% up and down but closed flat looks "calm" to it. That wastes information and makes a 5-day measurement noisy.

**Yang-Zhang** (a published method, 2000) uses three clues from every day:
- the **overnight gap**: yesterday's close to today's open,
- the **daytime drift**: today's open to today's close,
- the **high-low range** of the day.

More clues per day means a steadier number from the same 5 days. The formula blends the three clues; you do not need to memorise it, but read the comments in the code.

In [ ]:
# Cell 3: the Yang-Zhang wiggle
def yang_zhang(df, n=5):
    o, h, l, c = (np.log(df[col]) for col in ["Open", "High", "Low", "Close"])
    overnight = o - c.shift(1)                       # clue 1: gap from yesterday's close to today's open
    open_to_close = c - o                            # clue 2: what happened during the day
    range_clue = (h - c) * (h - o) + (l - c) * (l - o)   # clue 3: uses the day's high and low
    var_overnight = overnight.rolling(n).var()
    var_open_close = open_to_close.rolling(n).var()
    var_range = range_clue.rolling(n).mean()
    k = 0.34 / (1.34 + (n + 1) / (n - 1))            # the blending weight from the paper
    variance = var_overnight + k * var_open_close + (1 - k) * var_range
    return np.sqrt(variance) * 100                   # back to percent per day

yang_zhang(prices["RELIANCE"]).tail(3).round(2)

In [ ]:
# Cell 4: the two measures side by side on Reliance, last year
both = pd.DataFrame({
    "simple": simple_wiggle(prices["RELIANCE"]),
    "yang_zhang": yang_zhang(prices["RELIANCE"]),
}).dropna()
both.tail(250).plot(figsize=(10, 4), title="Reliance: wiggle by two methods (% per day)")
plt.ylabel("% per day")
plt.show()
print("How similar are they? correlation =", round(both["simple"].corr(both["yang_zhang"]), 2))

They move together, but the simple line is jumpier. That extra jumpiness is mostly **noise**: random flicker that has nothing to do with how risky the stock really is.

## Lining up the guess with the answer

Stand at the end of day **t**.
- Naive's guess uses only the **past**: the wiggle of days t-4 to t (all known today).
- The real answer is the **future**: the wiggle of days t+1 to t+5.

In pandas, `wiggle` on row t covers the 5 days *ending* at t. Moving it up by 5 rows with `shift(-5)` puts the "next 5 days" value on row t. We use that future value **only for scoring**, never as an input.

In [ ]:
# Cell 5: for every day, line up "what Naive would say" with "what really happened next"
def build_table(df):
    wiggle = yang_zhang(df, 5)
    table = pd.DataFrame({
        "naive_guess": wiggle,               # last 5 days (known today)
        "actual_next5": wiggle.shift(-5),    # next 5 days (the future; only for scoring)
    })
    return table.dropna()

tables = {s: build_table(df) for s, df in prices.items()}
tables["RELIANCE"].tail(8).round(2)

Notice the table **ends 5 days before the last price date**: for the most recent days, "the next 5 days" have not happened yet, so there is no answer to score against.

## Proof that we are not peeking

The next cell rebuilds the answer for one date **by hand**, using only the rows after that date (plus that day's close), and checks it matches the table.

In [ ]:
# Cell 6: check one date by hand
df = prices["RELIANCE"]
d = tables["RELIANCE"].index[-100]
i = df.index.get_loc(d)
by_hand = yang_zhang(df.iloc[i : i + 6], 5).iloc[-1]       # rows t .. t+5 only
in_table = tables["RELIANCE"].loc[d, "actual_next5"]
print("date:", d.date(), "| by hand:", round(by_hand, 4), "| in table:", round(in_table, 4))
assert np.isclose(by_hand, in_table), "mismatch: the table is not built the way we think"
print("They match.")

## The exam

We split by **date**: the most recent 3 years are the exam, and everything older is study material. Naive needs no studying, but later models will, and they must be scored on exactly this same exam.

The score is the **average miss**: how far the guess was from the real wiggle, averaged over every exam day, in percentage points of daily wiggle. Lower is better.

In [ ]:
# Cell 7: Naive's score on the exam
EXAM_YEARS = 3
rows = []
for s, t in tables.items():
    exam = t[t.index > t.index[-1] - pd.DateOffset(years=EXAM_YEARS)]
    miss = (exam["actual_next5"] - exam["naive_guess"]).abs()
    rows.append({
        "stock": s,
        "exam days": len(exam),
        "typical real wiggle (% a day)": exam["actual_next5"].median(),
        "Naive average miss (points)": miss.mean(),
    })
scores = pd.DataFrame(rows).set_index("stock")
scores["miss as share of wiggle"] = scores["Naive average miss (points)"] / scores["typical real wiggle (% a day)"]
scores.round(2)

How to read it: if the typical real wiggle is 1.2% a day and Naive misses by 0.4 points on average, then Naive is off by about a third. **That share is the number every later model has to beat.**

In [ ]:
# Cell 8: Naive's guess against what really happened, SBIN, exam period
t = tables["SBIN"]
exam = t[t.index > t.index[-1] - pd.DateOffset(years=EXAM_YEARS)]
exam.plot(figsize=(10, 4), title="SBIN: Naive guess vs the real next-5-day wiggle (% per day)")
plt.ylabel("% per day")
plt.show()

Look at the chart: the Naive line is simply the real line **arriving a week late**. It follows the ups and downs but always one step behind, so it is always wrong right at the turning points. A good model has to do better exactly there.

## Your turn

1. Change `EXAM_YEARS` in Cell 7 to 1, then to 5, and re-run Cells 7 and 8. Does Naive's miss change much?
2. Replace `"SBIN"` in Cell 8 with `"TCS"`. Which stock's wiggle looks calmer?
3. In Cell 5, look at the last row of `tables["RELIANCE"]`. Why does it stop 5 days before today?
4. Explain back, in one sentence: **why do we train on older dates and test on newer ones, instead of mixing dates randomly?**